In [1]:
# 导入依赖、设置路径、运行开关、embedding 配置和模型训练参数。
import ast
import hashlib
import json
import os
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import xgboost as xgb
from catboost import CatBoostClassifier
from IPython.display import display
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from openpyxl.styles import Font
from pytorch_tabnet.multitask import TabNetMultiTaskClassifier
from rtdl_revisiting_models import FTTransformer
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    coverage_error,
    f1_score,
    hamming_loss,
    label_ranking_average_precision_score,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_class_weight
from tab_transformer_pytorch import TabTransformer
from torch.utils.data import DataLoader, TensorDataset

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "1")

SOURCE_FILE = Path("data/ROTD_final.xlsx")
ROTD_FILE = Path("data/ROTD_preprocessed.xlsx")
SPLIT_FILE = Path("data/shared_train_test_split.csv")
RESULT_DIR = Path("results")
CACHE_DIR = Path("embedding_cache")
LLM_CONFIG_FILE = Path("LLM_config.json")

DIAGNOSIS_RESULT_FILE = RESULT_DIR / "use_case1_diagnosis.xlsx"

RANDOM_STATE = 42
TEST_SIZE = 0.3
MIN_SUPPORT = 2
REBUILD_ROTD = True
REBUILD_SPLIT = False
REBUILD_EMBEDDINGS = False

RUN_STRUCTURED_MODELS = False  # 当前只保留 fusion feature 结果，tabular-only 不单独训练。
RUN_EMBEDDING_MODELS = False  # 当前只保留 fusion feature 结果，embedding-only 不单独训练。
RUN_FUSION_MODELS = True
RUN_DEEP_MODELS = True

EMBED_MODELS = ["text-embedding-3-small", "text-embedding-3-large"]
FUSION_EMBED_MODELS = ["text-embedding-3-large"]
PCA_N_COMPONENTS_LIST = [32]
DIAGNOSIS_CACHE_TAG = "diagnosis_final_pastmed_reports_tumorsize"
LEGACY_DIAGNOSIS_CACHE_TAG = "diagnosis_v6_pastmed_reports_tumorsize"
EMBED_BATCH_SIZE = 32
MAX_RETRIES = 5
GENETIC_COLS = [
    "Genetic_testing_method",
    "Genotype",
    "Mutation_type",
    "Mutation_gene_name",
    "HGVS variant description",
    "Chromosome",
]

# 读取 LLM_config.json 中的 API key 和 base URL。
def load_llm_config(config_file=LLM_CONFIG_FILE):
    if not Path(config_file).exists():
        raise FileNotFoundError(f"Missing LLM config file: {config_file}")
    with Path(config_file).open("r", encoding="utf-8-sig") as fh:
        config = json.load(fh)
    api_key = config.get("api_key") or config.get("OPENAI_API_KEY") or config.get("key")
    base_url = config.get("base_url") or config.get("OPENAI_BASE_URL") or config.get("url")
    if not api_key:
        raise ValueError(f"{config_file} must include api_key or OPENAI_API_KEY")
    return api_key, base_url

OPENAI_API_KEY, OPENAI_BASE_URL = load_llm_config()

MLP_EPOCHS = 300
TABNET_EPOCHS = 200
TABTRANSFORMER_EPOCHS = 100
FTTRANSFORMER_EPOCHS = 400

RESULT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


## 共享数据准备

本节函数只构建一次诊断队列。后续所有模型都复用同一批病例、同一套疾病标签、同一段 embedding 输入文本，以及同一个训练/测试划分。


In [2]:
# 定义诊断数据清洗、ROTD 构建和共享训练测试划分相关函数。
INVALID_CODE_VALUES = {
    "NA_NOT_DISEASE",
    "NA_OTHER_DISEASE",
    "NA_NOT_PHENOTYPE",
    "NA_NOT_LOCATION",
    "NA_EMPTY",
    "",
    "[]",
    "nan",
    "None",
    "NaN",
}


# 清洗疾病、表型和部位等多值编码字段。
def clean_code_list(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if text in INVALID_CODE_VALUES:
        return []
    try:
        parsed = ast.literal_eval(text)
        values = parsed if isinstance(parsed, list) else [parsed]
    except (ValueError, SyntaxError):
        values = text.split(",")

    cleaned = []
    seen = set()
    for item in values:
        code = str(item).strip()
        if code in INVALID_CODE_VALUES or code in seen:
            continue
        cleaned.append(code)
        seen.add(code)
    return cleaned


# 将多值编码列展开为 0/1 宽表特征。
def expand_multivalue_codes(df, source_col):
    if source_col not in df.columns:
        return pd.DataFrame(index=df.index)
    code_lists = df[source_col].apply(clean_code_list)
    mlb = MultiLabelBinarizer()
    matrix = mlb.fit_transform(code_lists)
    return pd.DataFrame(matrix, columns=mlb.classes_, index=df.index)


# 将年、月、日形式的年龄字段统一换算成年。
def calculate_age_years(row):
    y_val = pd.to_numeric(row.get("Age_y"), errors="coerce")
    m_val = pd.to_numeric(row.get("Age_m"), errors="coerce")
    d_val = pd.to_numeric(row.get("Age_d"), errors="coerce")
    if pd.isna(y_val) and pd.isna(m_val) and pd.isna(d_val):
        return np.nan
    y_val = 0 if pd.isna(y_val) else y_val
    m_val = 0 if pd.isna(m_val) else m_val
    d_val = 0 if pd.isna(d_val) else d_val
    return y_val + m_val / 12 + d_val / 365


# 解析遗传信息字段中的列表字符串或单值文本。
def parse_genetic_list(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if text in {"", "[]", "nan", "None", "NaN"}:
        return []
    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        values = text.replace(";", ",").split(",")
        return sorted({item.strip() for item in values if item.strip()})
    if isinstance(parsed, (list, tuple, set)):
        return sorted({str(item).strip() for item in parsed if str(item).strip()})
    return [str(parsed).strip()] if str(parsed).strip() else []


# 统一清洗文本字段中的空值和占位符。
def normalize_text_value(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    text = str(value).strip()
    return "" if text in {"", "[]", "nan", "None", "NaN"} else text


# 拼接病史、检查报告和肿瘤大小，生成 embedding 输入文本。
def build_diagnosis_text(row):
    past_med_hist = (
        normalize_text_value(row.get("Past_medical_history"))
        or "No past medical history (diseases/surgeries/medications/family history) mentioned"
    )
    reports = normalize_text_value(row.get("Clinical_reports")) or "N/A"
    tumor_size = normalize_text_value(row.get("Tumor_size")) or "N/A"
    return f"past_med_hist:\n{past_med_hist}\n\nreports:\n{reports}\n\ntumor_size:\n{tumor_size}"


# 找出展开后的 ORPHA 和 ICD-11 疾病标签列。
def disease_label_columns(df):
    return [col for col in df.columns if col.startswith("ORPHA") or col.startswith("ICD-11")]


# 从人工校正主表生成诊断建模用的 ROTD_preprocessed 中间表。
def prepare_rotd(source_file=SOURCE_FILE, output_file=ROTD_FILE, min_support=MIN_SUPPORT):
    df = pd.read_excel(source_file, header=0, dtype=str)
    required = [
        "pat_id", "Disease_ids", "Phenotype_ids", "Anatomical_location_ids",
        "Age_y", "Age_m", "Age_d", "Gender", "Ethnicity", "Country", "Laterality",
        "Past_medical_history", "Clinical_reports", "Tumor_size",
        *GENETIC_COLS,
    ]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required source columns: {missing}")

    for col in GENETIC_COLS:
        df[col] = df[col].replace("[]", pd.NA)

    pheno_df = expand_multivalue_codes(df, "Phenotype_ids")
    loc_df = expand_multivalue_codes(df, "Anatomical_location_ids")
    disease_df = expand_multivalue_codes(df, "Disease_ids")
    data = pd.concat([df, pheno_df, loc_df, disease_df], axis=1)
    data["age"] = data.apply(calculate_age_years, axis=1)

    label_cols = disease_label_columns(data)
    if not label_cols:
        raise ValueError("No expanded ORPHA or ICD-11 disease columns were created.")
    data[label_cols] = data[label_cols].fillna(0).astype(int)
    support = data[label_cols].sum(axis=0)
    valid_labels = support[support >= min_support].index.tolist()
    if not valid_labels:
        raise ValueError(f"No disease labels meet MIN_SUPPORT={min_support}.")

    before_shape = data.shape
    data = data.loc[data[valid_labels].sum(axis=1) > 0].reset_index(drop=True)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    data.to_excel(output_file, index=False)

    print(f"Source rows/columns: {df.shape}")
    print(f"Expanded phenotype/location/disease columns: {pheno_df.shape[1]}/{loc_df.shape[1]}/{disease_df.shape[1]}")
    print(f"ROTD before/after label filtering: {before_shape} -> {data.shape}")
    print(f"Retained disease labels: {len(valid_labels)}")
    print(f"Saved {output_file}")
    return data, valid_labels


# 读取 ROTD 中间表并构建疾病标签矩阵。
def load_diagnosis_data(rotd_file=ROTD_FILE, min_support=MIN_SUPPORT):
    data = pd.read_excel(rotd_file, sheet_name="Sheet1", header=0, dtype=str)
    label_cols = disease_label_columns(data)
    if not label_cols:
        raise ValueError("No expanded disease label columns found. Rebuild ROTD_preprocessed.xlsx first.")
    data[label_cols] = data[label_cols].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
    support = data[label_cols].sum(axis=0)
    valid_labels = support[support >= min_support].index.tolist()
    data = data.loc[data[valid_labels].sum(axis=1) > 0].reset_index(drop=True)
    y = data[valid_labels].to_numpy(dtype=int)

    required_text_cols = ["pat_id", "Past_medical_history", "Clinical_reports", "Tumor_size"]
    missing = [col for col in required_text_cols if col not in data.columns]
    if missing:
        raise ValueError(f"Missing required diagnosis columns: {missing}")
    data["raw_input_text"] = data.apply(build_diagnosis_text, axis=1)
    return data, y, valid_labels


# 基于多标签疾病矩阵创建训练集和测试集划分。
def create_split_frame(data, y):
    splitter = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    train_idx, test_idx = next(splitter.split(data, y))
    split_df = pd.DataFrame({"pat_id": data["pat_id"].astype(str), "split": "train"})
    split_df.loc[test_idx, "split"] = "test"
    return split_df


# 读取或重建共享 train/test split，并校验 pat_id 一致性。
def load_or_create_split(data, y, split_file=SPLIT_FILE, rebuild=REBUILD_SPLIT):
    split_file = Path(split_file)
    current_ids = data["pat_id"].astype(str)
    if current_ids.duplicated().any():
        duplicated = current_ids[current_ids.duplicated()].unique()[:10].tolist()
        raise ValueError(f"Duplicate pat_id values in current data: {duplicated}")
    if len(data) != y.shape[0]:
        raise ValueError(f"data/y row count mismatch: {len(data)} vs {y.shape[0]}")

    if rebuild or not split_file.exists():
        split_df = create_split_frame(data, y)
        split_file.parent.mkdir(parents=True, exist_ok=True)
        split_df.to_csv(split_file, index=False)
        print(f"Saved shared split to {split_file}")
    else:
        split_df = pd.read_csv(split_file, dtype={"pat_id": str, "split": str})

    if split_df["pat_id"].duplicated().any():
        duplicated = split_df.loc[split_df["pat_id"].duplicated(), "pat_id"].unique()[:10].tolist()
        raise ValueError(f"Duplicate pat_id values in shared split file: {duplicated}")
    invalid_split = sorted(set(split_df["split"]) - {"train", "test"})
    if invalid_split:
        raise ValueError(f"Shared split contains invalid labels: {invalid_split}")

    expected_ids = set(current_ids)
    saved_ids = set(split_df["pat_id"])
    if expected_ids != saved_ids:
        raise ValueError(
            "Shared split does not match current patients. "
            f"Missing examples: {sorted(expected_ids - saved_ids)[:10]}; "
            f"extra examples: {sorted(saved_ids - expected_ids)[:10]}."
        )

    split_labels = current_ids.map(split_df.set_index("pat_id")["split"])
    train_idx = np.flatnonzero(split_labels.eq("train").to_numpy())
    test_idx = np.flatnonzero(split_labels.eq("test").to_numpy())
    train_empty = np.flatnonzero(y[train_idx].sum(axis=0) == 0)
    test_empty = np.flatnonzero(y[test_idx].sum(axis=0) == 0)
    if len(train_empty) or len(test_empty):
        raise ValueError(
            "共享 split 未能保证每个保留疾病标签在 train/test 中都有阳性病例。"
            f" train empty labels: {len(train_empty)}, test empty labels: {len(test_empty)}."
        )
    return train_idx, test_idx


## 共享建模工具

本节统一定义评估指标、阈值搜索、传统机器学习模型和深度学习模型。结构化特征、文本 embedding 和多模态 fusion 都调用同一个训练入口。


In [3]:
# 定义通用评估指标、阈值搜索和模型训练函数。
METRIC_COLS = [
    "hamming_loss",
    "exact_match",
    "micro_f1",
    "macro_f1",
    "top1",
    "top3",
    "top5",
    "lrap",
    "coverage",
]
LOWER_IS_BETTER = {"hamming_loss", "coverage"}


# 创建兼容不同 scikit-learn 版本的 one-hot 编码器。
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


# 清洗特征名中的 XGBoost 禁用字符，并确保 DataFrame 列名唯一。
def make_unique_feature_names(names):
    seen = {}
    unique = []
    for raw_name in names:
        name = str(raw_name)
        name = (
            name.replace("[", "(")
            .replace("]", ")")
            .replace("<", "lt_")
            .replace("\n", " ")
            .replace("\r", " ")
            .strip()
        )
        if not name:
            name = "feature"
        count = seen.get(name, 0)
        unique.append(name if count == 0 else f"{name}__{count}")
        seen[name] = count + 1
    return unique


# 将不同模型的概率或决策分数统一整理成标签得分矩阵。
def get_score_matrix(model, X_input):
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_input)
        if isinstance(proba, list):
            score_cols = []
            for item in proba:
                if item.ndim == 2 and item.shape[1] > 1:
                    score_cols.append(item[:, 1])
                else:
                    score_cols.append(np.zeros(len(X_input), dtype=float))
            return np.column_stack(score_cols)
        if proba.ndim == 3:
            return proba[:, :, 1]
        return proba
    if hasattr(model, "decision_function"):
        scores = np.asarray(model.decision_function(X_input))
        return scores if scores.ndim == 2 else scores.reshape(-1, 1)
    return np.asarray(model.predict(X_input), dtype=float)


# 计算至少命中一个真实标签的 top-k 准确率。
def top_k_accuracy(y_true, score_matrix, k):
    top_k_idx = np.argsort(score_matrix, axis=1)[:, -k:]
    hits = [y_true[i, idx].max() for i, idx in enumerate(top_k_idx)]
    return float(np.mean(hits))


# 在训练集上为每个疾病标签搜索 F1 最优阈值。
def get_optimal_thresholds(y_true, y_score):
    thresholds = []
    for i in range(y_true.shape[1]):
        best_t, best_f1 = 0.5, 0
        for threshold in np.linspace(0.001, 0.999, 199):
            pred = (y_score[:, i] >= threshold).astype(int)
            score = f1_score(y_true[:, i], pred, zero_division=0)
            if score > best_f1:
                best_t, best_f1 = threshold, score
        thresholds.append(best_t)
    return np.asarray(thresholds)


# 根据二值预测和得分矩阵计算所有诊断评估指标。
def evaluate_predictions(y_true, y_pred, score_matrix, model_name, metadata):
    y_pred = np.asarray(y_pred, dtype=int)
    score_matrix = np.asarray(score_matrix, dtype=float)
    result = {
        "model": model_name,
        "hamming_loss": hamming_loss(y_true, y_pred),
        "exact_match": accuracy_score(y_true, y_pred),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "top1": top_k_accuracy(y_true, score_matrix, 1),
        "top3": top_k_accuracy(y_true, score_matrix, min(3, score_matrix.shape[1])),
        "top5": top_k_accuracy(y_true, score_matrix, min(5, score_matrix.shape[1])),
        "lrap": label_ranking_average_precision_score(y_true, score_matrix),
        "coverage": coverage_error(y_true, score_matrix),
    }
    return {**metadata, **result}


# 固定随机种子，尽量提高模型训练的可复现性。
def set_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# 定义所有传统机器学习模型及其参数。
def sklearn_model_factories():
    return [
        (
            "RandomForest",
            lambda: MultiOutputClassifier(
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=15,
                    min_samples_leaf=2,
                    class_weight="balanced_subsample",
                    max_features="sqrt",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )
            ),
        ),
        (
            "XGBoost",
            lambda: MultiOutputClassifier(
                xgb.XGBClassifier(
                    objective="binary:logistic",
                    n_estimators=500,
                    learning_rate=0.005,
                    max_depth=2,
                    subsample=0.6,
                    colsample_bytree=0.6,
                    reg_alpha=1.0,
                    reg_lambda=2.0,
                    random_state=RANDOM_STATE,
                    tree_method="hist",
                )
            ),
        ),
        (
            "LogisticRegression",
            lambda: OneVsRestClassifier(
                LogisticRegression(
                    max_iter=5000,
                    C=0.1,
                    class_weight="balanced",
                    solver="liblinear",
                    random_state=RANDOM_STATE,
                ),
                n_jobs=-1,
            ),
        ),
        (
            "CatBoost",
            lambda: MultiOutputClassifier(
                CatBoostClassifier(
                    iterations=500,
                    learning_rate=0.01,
                    depth=4,
                    l2_leaf_reg=5.0,
                    subsample=0.6,
                    auto_class_weights="Balanced",
                    random_state=RANDOM_STATE,
                    verbose=0,
                )
            ),
        ),
        (
            "SVC",
            lambda: MultiOutputClassifier(
                SVC(
                    kernel="linear",
                    C=0.01,
                    probability=True,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
                n_jobs=-1,
            ),
        ),
    ]


# 定义用于多标签诊断的全连接神经网络。
class PyTorchMLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, out_dim),
        )

    def forward(self, x):
        return self.layers(x)


# 定义仅使用连续特征的 TabTransformer 包装模型。
class DummyTabTransformerModel(nn.Module):
    def __init__(self, num_continuous, num_classes):
        super().__init__()
        self.model = TabTransformer(
            categories=(1,),
            num_continuous=num_continuous,
            dim=16,
            depth=2,
            heads=2,
            dim_out=num_classes,
            mlp_hidden_mults=(2,),
            mlp_act=nn.ReLU(),
            attn_dropout=0.3,
            ff_dropout=0.3,
        )

    def forward(self, x_cat, x_cont):
        return self.model(x_cat, x_cont)


# 构建 PyTorch 张量、DataLoader 和类别不平衡损失函数。
def make_torch_training_data(X_train, y_train, batch_size=32):
    set_seed(RANDOM_STATE)
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)

    classes = np.arange(2)
    pos_weights = []
    for i in range(y_train.shape[1]):
        try:
            weight = compute_class_weight("balanced", classes=classes, y=y_train[:, i])[1]
        except ValueError:
            weight = 1.0
        pos_weights.append(weight)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weights, dtype=torch.float32).to(device))
    return X_train_tensor, y_train_tensor, train_loader, criterion


# 训练 PyTorch 模型，并可选使用学习率调度器。
def train_torch_model(model, train_loader, criterion, optimizer, epochs, forward_fn=None, scheduler=None):
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            if len(batch) == 2:
                xb, yb = batch
                pred = forward_fn(model, xb) if forward_fn is not None else model(xb)
            else:
                *features, yb = batch
                pred = model(*features)
            loss = criterion(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        if scheduler is not None:
            scheduler.step()
        if (epoch + 1) % 10 == 0:
            lr = optimizer.param_groups[0]["lr"]
            print(f"Epoch {epoch + 1:3d} | LR: {lr:.6f} | Loss: {total_loss / len(train_loader):.4f}")
    model.eval()
    return model


# 使用训练集阈值将测试集得分转为 0/1 预测并保存指标。
def append_thresholded_result(results, y_train, y_test, train_score, test_score, model_name, metadata):
    thresholds = get_optimal_thresholds(y_train, train_score)
    pred = (test_score >= thresholds).astype(int)
    results.append(evaluate_predictions(y_test, pred, test_score, model_name, metadata))


# 对同一组特征统一训练传统模型和深度模型。
def run_model_suite(feature_set, X_train, X_test, feature_names, y_train, y_test, metadata=None, run_deep=RUN_DEEP_MODELS):
    metadata = {"feature_set": feature_set, **(metadata or {})}
    feature_names = make_unique_feature_names(feature_names or [f"f_{i}" for i in range(X_train.shape[1])])
    X_train = np.asarray(X_train, dtype=np.float32)
    X_test = np.asarray(X_test, dtype=np.float32)
    X_train_df = pd.DataFrame(X_train, columns=feature_names)
    X_test_df = pd.DataFrame(X_test, columns=feature_names)
    results = []

    print(f"===== {feature_set}: {X_train.shape[1]} features, {y_train.shape[1]} labels =====")
    for model_name, make_model in sklearn_model_factories():
        print(f"Training {model_name}...")
        model = make_model()
        model.fit(X_train_df, y_train)
        train_score = get_score_matrix(model, X_train_df)
        test_score = get_score_matrix(model, X_test_df)
        append_thresholded_result(results, y_train, y_test, train_score, test_score, model_name, metadata)

    if not run_deep:
        return pd.DataFrame(results)

    num_features = X_train.shape[1]
    num_classes = y_train.shape[1]
    X_train_tensor, y_train_tensor, train_loader, criterion = make_torch_training_data(X_train, y_train)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

    print("Training MLP...")
    mlp = PyTorchMLP(num_features, num_classes).to(device)
    optimizer = optim.AdamW(mlp.parameters(), lr=5e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
    train_torch_model(mlp, train_loader, criterion, optimizer, epochs=MLP_EPOCHS, scheduler=scheduler)
    with torch.no_grad():
        train_score = torch.sigmoid(mlp(X_train_tensor)).cpu().numpy()
        test_score = torch.sigmoid(mlp(X_test_tensor)).cpu().numpy()
    append_thresholded_result(results, y_train, y_test, train_score, test_score, "MLP", metadata)

    print("Training TabNet...")
    tabnet = TabNetMultiTaskClassifier(
        seed=0,  # Explicit library default; preserve the existing benchmark.
        n_d=16,
        n_a=16,
        n_steps=3,
        gamma=1.2,
        n_independent=1,
        n_shared=1,
        cat_idxs=[],
        cat_emb_dim=1,
        lambda_sparse=5e-4,
        momentum=0.02,
        mask_type="sparsemax",
        verbose=10,
    )
    tabnet.fit(X_train, y_train, max_epochs=TABNET_EPOCHS, batch_size=64, virtual_batch_size=16)
    train_score = np.column_stack([prob[:, 1] for prob in tabnet.predict_proba(X_train)])
    test_score = np.column_stack([prob[:, 1] for prob in tabnet.predict_proba(X_test)])
    append_thresholded_result(results, y_train, y_test, train_score, test_score, "TabNet", metadata)

    print("Training TabTransformer...")
    x_cat_train = torch.zeros((X_train.shape[0], 1), dtype=torch.long).to(device)
    x_cat_test = torch.zeros((X_test.shape[0], 1), dtype=torch.long).to(device)
    tab_loader = DataLoader(TensorDataset(x_cat_train, X_train_tensor, y_train_tensor), batch_size=16, shuffle=True)
    tab_trans = DummyTabTransformerModel(num_features, num_classes).to(device)
    optimizer = optim.AdamW(tab_trans.parameters(), lr=1e-4, weight_decay=1e-4)
    train_torch_model(tab_trans, tab_loader, criterion, optimizer, epochs=TABTRANSFORMER_EPOCHS)
    with torch.no_grad():
        train_score = torch.sigmoid(tab_trans(x_cat_train, X_train_tensor)).cpu().numpy()
        test_score = torch.sigmoid(tab_trans(x_cat_test, X_test_tensor)).cpu().numpy()
    append_thresholded_result(results, y_train, y_test, train_score, test_score, "TabTransformer", metadata)

    print("Training FTTransformer...")
    ft_trans = FTTransformer(
        n_cont_features=num_features,
        cat_cardinalities=[],
        d_out=num_classes,
        n_blocks=1,
        d_block=16,
        attention_n_heads=4,
        ffn_d_hidden_multiplier=2,
        ffn_activation="ReGLU",
        attention_dropout=0.1,
        ffn_dropout=0.1,
        residual_dropout=0.0,
    ).to(device)
    optimizer = optim.AdamW(ft_trans.parameters(), lr=3e-3, weight_decay=2e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=400, eta_min=1e-5)
    train_torch_model(
        ft_trans,
        train_loader,
        criterion,
        optimizer,
        epochs=FTTRANSFORMER_EPOCHS,
        forward_fn=lambda model, xb: model(xb, None),
        scheduler=scheduler,
    )
    with torch.no_grad():
        train_score = torch.sigmoid(ft_trans(X_train_tensor, None)).cpu().numpy()
        test_score = torch.sigmoid(ft_trans(X_test_tensor, None)).cpu().numpy()
    append_thresholded_result(results, y_train, y_test, train_score, test_score, "FTTransformer", metadata)

    return pd.DataFrame(results)


## Embedding 工具

Embedding 缓存会同时记录模型名称和文本哈希，因此不会把旧 v5 文本矩阵误用于当前 v6 数据。


In [4]:
# 定义 embedding 缓存、校验、生成和 OpenAI 客户端相关函数。
# 读取 JSON 缓存文件，文件不存在时返回空字典。
def load_json(path):
    path = Path(path)
    if path.exists():
        with path.open("r", encoding="utf-8") as fh:
            return json.load(fh)
    return {}


# 保存 JSON 缓存文件。
def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        json.dump(payload, fh, ensure_ascii=False, indent=2)


# 计算文本列表的哈希，用于校验 embedding 缓存是否匹配当前数据。
def hash_texts(texts):
    hasher = hashlib.sha256()
    for text in texts:
        encoded = str(text).encode("utf-8")
        hasher.update(len(encoded).to_bytes(8, "little"))
        hasher.update(encoded)
    return hasher.hexdigest()


# 根据 LLM_config.json 创建 OpenAI 兼容客户端。
def get_openai_client():
    if OpenAI is None:
        raise ImportError("The openai package is not installed.")
    client_kwargs = {"api_key": OPENAI_API_KEY}
    if OPENAI_BASE_URL:
        client_kwargs["base_url"] = OPENAI_BASE_URL
    return OpenAI(**client_kwargs)


# 生成指定 embedding 模型对应的缓存文件路径。
def embedding_paths(model_name):
    safe_model = model_name.replace("/", "_")
    return {
        "matrix": CACHE_DIR / f"embedding_matrix_{DIAGNOSIS_CACHE_TAG}_{safe_model}.npy",
        "meta": CACHE_DIR / f"embedding_matrix_{DIAGNOSIS_CACHE_TAG}_{safe_model}.meta.json",
        "json": CACHE_DIR / f"embedding_cache_{DIAGNOSIS_CACHE_TAG}_{safe_model}.json",
    }


# 读取 embedding 矩阵，并用 meta 信息校验模型和文本哈希。
def load_embedding_matrix(matrix_path, meta_path, texts, model_name):
    matrix_path = Path(matrix_path)
    meta_path = Path(meta_path)
    if REBUILD_EMBEDDINGS or not matrix_path.exists():
        return None

    matrix = np.load(matrix_path).astype(np.float32)
    if matrix.shape[0] != len(texts):
        raise ValueError(f"{matrix_path} row count mismatch: {matrix.shape[0]} vs {len(texts)}")

    if meta_path.exists():
        meta = load_json(meta_path)
        expected_hash = hash_texts(texts)
        if meta.get("model_name") != model_name:
            raise ValueError(f"Embedding model mismatch in {meta_path}")
        if meta.get("text_hash") != expected_hash:
            print(f"Embedding text hash changed in {meta_path}; rebuilding from text cache.")
            return None
    else:
        print(f"Warning: missing {meta_path}; using row-count validation only.")

    print(f"Loaded embedding matrix from {matrix_path}")
    return matrix


# 保存 embedding 矩阵及其 meta 校验信息。
def save_embedding_matrix(matrix_path, meta_path, matrix, texts, model_name):
    np.save(matrix_path, matrix.astype(np.float32))
    save_json(
        meta_path,
        {
            "model_name": model_name,
            "text_count": len(texts),
            "text_hash": hash_texts(texts),
            "shape": list(matrix.shape),
            "dtype": str(matrix.dtype),
        },
    )


# 批量调用 embedding API，并优先复用逐条文本缓存。
def generate_embeddings(texts, model_name, cache_path):
    cache = load_json(cache_path)
    if not cache:
        safe_model = model_name.replace("/", "_")
        legacy_path = CACHE_DIR / f"embedding_cache_{LEGACY_DIAGNOSIS_CACHE_TAG}_{safe_model}.json"
        cache = load_json(legacy_path)
        if cache:
            print(f"Reusing {len(cache)} cached texts from {legacy_path}")
    vectors = []
    missing_texts = []
    missing_indices = []
    for idx, text in enumerate(texts):
        cached = cache.get(text)
        if cached is None:
            vectors.append(None)
            missing_texts.append(text)
            missing_indices.append(idx)
        else:
            vectors.append(np.asarray(cached, dtype=np.float32))

    client = None
    for start in range(0, len(missing_texts), EMBED_BATCH_SIZE):
        if client is None:
            client = get_openai_client()
        batch_texts = missing_texts[start : start + EMBED_BATCH_SIZE]
        batch_indices = missing_indices[start : start + EMBED_BATCH_SIZE]
        last_error = None
        for attempt in range(MAX_RETRIES):
            try:
                response = client.embeddings.create(model=model_name, input=batch_texts)
                break
            except Exception as exc:
                last_error = exc
                time.sleep(min(2**attempt, 10))
        else:
            raise RuntimeError(f"Failed to create embeddings for {model_name}: {last_error}")

        for target_idx, item in zip(batch_indices, response.data):
            vector = np.asarray(item.embedding, dtype=np.float32)
            vectors[target_idx] = vector
            cache[texts[target_idx]] = vector.tolist()
        save_json(cache_path, cache)

    if not Path(cache_path).exists():
        save_json(cache_path, cache)
    return np.vstack(vectors).astype(np.float32)


# 读取已有 embedding 矩阵，不存在时调用 API 生成。
def load_or_create_embeddings(data, model_names):
    texts = data["raw_input_text"].tolist()
    matrices = {}
    for model_name in model_names:
        paths = embedding_paths(model_name)
        matrix = load_embedding_matrix(paths["matrix"], paths["meta"], texts, model_name)
        if matrix is None:
            print(f"Generating embeddings: {model_name}")
            matrix = generate_embeddings(texts, model_name, paths["json"])
            save_embedding_matrix(paths["matrix"], paths["meta"], matrix, texts, model_name)
            print(f"Saved embedding matrix to {paths['matrix']}")
        matrices[model_name] = matrix
        print(f"{model_name}: {matrix.shape}")
    return matrices


## 构建诊断队列、划分和结构化特征


In [5]:
# 生成或读取诊断队列，并创建共享训练测试划分。
if REBUILD_ROTD or not ROTD_FILE.exists():
    prepare_rotd(SOURCE_FILE, ROTD_FILE, MIN_SUPPORT)

data, y, valid_disease_cols = load_diagnosis_data(ROTD_FILE, MIN_SUPPORT)
train_idx, test_idx = load_or_create_split(data, y, SPLIT_FILE, REBUILD_SPLIT)
y_train = y[train_idx]
y_test = y[test_idx]

print(f"Diagnosis samples: {len(data)}")
print(f"Train/test: {len(train_idx)}/{len(test_idx)}")
print(f"Disease labels: {len(valid_disease_cols)}")
print(f"Train empty labels: {(y_train.sum(axis=0) == 0).sum()}, test empty labels: {(y_test.sum(axis=0) == 0).sum()}")
display(data[["pat_id", "raw_input_text"]].head(3))


Source rows/columns: (2134, 40)
Expanded phenotype/location/disease columns: 1039/666/350
ROTD before/after label filtering: (2134, 2096) -> (2056, 2096)
Retained disease labels: 147
Saved data\ROTD_preprocessed.xlsx


Diagnosis samples: 2056
Train/test: 1426/630
Disease labels: 147
Train empty labels: 0, test empty labels: 0


,pat_id,raw_input_text
0,pat_26077847_01,past_med_hist:\nCataract surgery (no improveme...
1,pat_26077847_02,past_med_hist:\nNo past medical history (disea...
2,pat_26869128_01,past_med_hist:\nNo past medical history (disea...


In [6]:
# 构建所有模型共用的结构化 tabular 特征矩阵。
# 构建结构化特征矩阵，包括人口学、表型、部位和遗传特征。
def build_tabular_features(data, train_idx, test_idx):
    frame = data.copy()
    frame["age"] = frame.apply(calculate_age_years, axis=1) if "age" not in frame.columns else pd.to_numeric(frame["age"], errors="coerce")
    train_df = frame.iloc[train_idx].reset_index(drop=True)
    test_df = frame.iloc[test_idx].reset_index(drop=True)

    age_imputer = SimpleImputer(strategy="median")
    age_scaler = StandardScaler()
    X_train_age = age_scaler.fit_transform(age_imputer.fit_transform(train_df[["age"]].values))
    X_test_age = age_scaler.transform(age_imputer.transform(test_df[["age"]].values))

    cat_cols = ["Gender", "Ethnicity", "Country", "Laterality"]
    for col in cat_cols:
        if col not in train_df.columns:
            train_df[col] = "Unknown"
            test_df[col] = "Unknown"
    train_df[cat_cols] = train_df[cat_cols].fillna("Unknown")
    test_df[cat_cols] = test_df[cat_cols].fillna("Unknown")
    ohe = make_one_hot_encoder()
    X_train_cat = ohe.fit_transform(train_df[cat_cols])
    X_test_cat = ohe.transform(test_df[cat_cols])

    pheno_cols = [col for col in frame.columns if col.startswith(("HP", "NA_OTHER_PHENOTYPE"))]
    loc_cols = [col for col in frame.columns if col.startswith(("SCTID", "NA_OTHER_LOCATION"))]
    X_train_pheno = train_df[pheno_cols].fillna(0).astype(float).to_numpy(np.float32)
    X_test_pheno = test_df[pheno_cols].fillna(0).astype(float).to_numpy(np.float32)
    X_train_loc = train_df[loc_cols].fillna(0).astype(float).to_numpy(np.float32)
    X_test_loc = test_df[loc_cols].fillna(0).astype(float).to_numpy(np.float32)

    genetic_matrices_train = []
    genetic_matrices_test = []
    genetic_feature_names = []
    missing_genetic_cols = [col for col in GENETIC_COLS if col not in frame.columns]
    if missing_genetic_cols:
        raise ValueError(f"缺少必需遗传特征列: {missing_genetic_cols}")
    for col in GENETIC_COLS:
        value_lists = frame[col].apply(parse_genetic_list)
        classes = sorted({value for values in value_lists for value in values})
        if not classes:
            continue
        mlb = MultiLabelBinarizer(classes=classes)
        genetic_matrices_train.append(mlb.fit_transform(train_df[col].apply(parse_genetic_list)))
        genetic_matrices_test.append(mlb.transform(test_df[col].apply(parse_genetic_list)))
        genetic_feature_names.extend([f"{col}::{value}" for value in classes])

    if genetic_matrices_train:
        X_train_genetic = np.hstack(genetic_matrices_train).astype(np.float32)
        X_test_genetic = np.hstack(genetic_matrices_test).astype(np.float32)
    else:
        X_train_genetic = np.empty((len(train_df), 0), dtype=np.float32)
        X_test_genetic = np.empty((len(test_df), 0), dtype=np.float32)

    X_train = np.hstack([X_train_age, X_train_cat, X_train_pheno, X_train_loc, X_train_genetic]).astype(np.float32)
    X_test = np.hstack([X_test_age, X_test_cat, X_test_pheno, X_test_loc, X_test_genetic]).astype(np.float32)
    feature_names = (
        ["age"]
        + list(ohe.get_feature_names_out(cat_cols))
        + pheno_cols
        + loc_cols
        + genetic_feature_names
    )
    print(f"Tabular features: {X_train.shape}/{X_test.shape}")
    return X_train, X_test, feature_names


X_train_tabular, X_test_tabular, tabular_feature_names = build_tabular_features(data, train_idx, test_idx)


Tabular features: (1426, 2747)/(630, 2747)


## 1. 结构化特征诊断模型


In [7]:
# 训练结构化特征诊断模型，并在 notebook 中展示结果。
structured_results = pd.DataFrame()
if RUN_STRUCTURED_MODELS:
    structured_results = run_model_suite(
        "tabular",
        X_train_tabular,
        X_test_tabular,
        tabular_feature_names,
        y_train,
        y_test,
    )
    display(structured_results.sort_values("micro_f1", ascending=False).round(4))


## 2. 文本 Embedding 诊断模型


In [8]:
# 生成或读取文本 embedding；当前不单独训练 embedding-only，只为 fusion 准备需要的 embedding。
embedding_results = pd.DataFrame()
embedding_matrices = {}

# 根据启用的任务决定需要加载哪些 embedding 模型，避免只跑 fusion 时额外生成 embedding-only 缓存。
embedding_models_needed = []
if RUN_EMBEDDING_MODELS:
    embedding_models_needed.extend(EMBED_MODELS)
if RUN_FUSION_MODELS:
    embedding_models_needed.extend(FUSION_EMBED_MODELS)
embedding_models_needed = list(dict.fromkeys(embedding_models_needed))

if embedding_models_needed:
    embedding_matrices = load_or_create_embeddings(data, embedding_models_needed)

if RUN_EMBEDDING_MODELS:
    embedding_frames = []
    for embedding_model, X_full in embedding_matrices.items():
        scaler = StandardScaler()
        X_train_embedding = scaler.fit_transform(X_full[train_idx]).astype(np.float32)
        X_test_embedding = scaler.transform(X_full[test_idx]).astype(np.float32)
        feature_names = [f"{embedding_model}_e{i + 1}" for i in range(X_train_embedding.shape[1])]
        result = run_model_suite(
            "embedding",
            X_train_embedding,
            X_test_embedding,
            feature_names,
            y_train,
            y_test,
            metadata={"text_variant": "raw_input_text", "embedding_model": embedding_model},
        )
        embedding_frames.append(result)

    embedding_results = pd.concat(embedding_frames, ignore_index=True)
    display(embedding_results.sort_values(["embedding_model", "micro_f1"], ascending=[True, False]).round(4))


Embedding text hash changed in embedding_cache\embedding_matrix_diagnosis_final_pastmed_reports_tumorsize_text-embedding-3-large.meta.json; rebuilding from text cache.
Generating embeddings: text-embedding-3-large


Saved embedding matrix to embedding_cache\embedding_matrix_diagnosis_final_pastmed_reports_tumorsize_text-embedding-3-large.npy
text-embedding-3-large: (2056, 3072)


## 3. 多模态 Early-Fusion 诊断模型


In [9]:
# 将结构化特征与 PCA 后的文本 embedding 拼接，训练 early-fusion 模型。
fusion_results = pd.DataFrame()

if RUN_FUSION_MODELS:
    fusion_frames = []
    for embedding_model in FUSION_EMBED_MODELS:
        X_full = embedding_matrices[embedding_model]
        embed_scaler = StandardScaler()
        X_train_embedding = embed_scaler.fit_transform(X_full[train_idx])
        X_test_embedding = embed_scaler.transform(X_full[test_idx])
        max_components = min(X_train_embedding.shape[0] - 1, X_train_embedding.shape[1])

        for requested_components in PCA_N_COMPONENTS_LIST:
            n_components = min(requested_components, max_components)
            pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
            # Fixed BLAS threads preserve the PCA inputs of the 2026-10-28 benchmark.
            with threadpool_limits(limits=1, user_api="blas"):
                X_train_pca = pca.fit_transform(X_train_embedding).astype(np.float32)
                X_test_pca = pca.transform(X_test_embedding).astype(np.float32)
            X_train_fusion = np.hstack([X_train_tabular, X_train_pca]).astype(np.float32)
            X_test_fusion = np.hstack([X_test_tabular, X_test_pca]).astype(np.float32)
            feature_names = tabular_feature_names + [f"{embedding_model}_pc{i + 1}" for i in range(n_components)]
            explained = float(pca.explained_variance_ratio_.sum())

            print(
                f"Fusion {embedding_model} | requested PCA={requested_components}, "
                f"used PCA={n_components}, explained variance={explained:.3f}"
            )
            result = run_model_suite(
                "fusion",
                X_train_fusion,
                X_test_fusion,
                feature_names,
                y_train,
                y_test,
                metadata={
                    "text_variant": "raw_input_text",
                    "embedding_model": embedding_model,
                    "requested_pca_components": requested_components,
                    "pca_components": n_components,
                    "pca_explained_variance": explained,
                },
            )
            fusion_frames.append(result)

    fusion_results = pd.concat(fusion_frames, ignore_index=True)
    display(fusion_results.sort_values(["requested_pca_components", "micro_f1"], ascending=[True, False]).round(4))


Fusion text-embedding-3-large | requested PCA=32, used PCA=32, explained variance=0.591
===== fusion: 2779 features, 147 labels =====
Training RandomForest...


Training XGBoost...


Training LogisticRegression...


Training CatBoost...


Training SVC...


Training MLP...


... (100 outputs left out)

Epoch 380 | LR: 0.000028 | Loss: 0.0545


Epoch 390 | LR: 0.000015 | Loss: 0.0602


Epoch 400 | LR: 0.000010 | Loss: 0.0570


,feature_set,text_variant,embedding_model,requested_pca_components,pca_components,pca_explained_variance,model,hamming_loss,exact_match,micro_f1,macro_f1,top1,top3,top5,lrap,coverage
0,fusion,raw_input_text,text-embedding-3-large,32,32,0.5909,RandomForest,0.0038,0.6254,0.7440,0.3092,0.7873,0.8651,0.8984,0.8057,7.1810
3,fusion,raw_input_text,text-embedding-3-large,32,32,0.5909,CatBoost,0.0041,0.6159,0.7356,0.3406,0.8159,0.9143,0.9413,0.8376,6.2619
2,fusion,raw_input_text,text-embedding-3-large,32,32,0.5909,LogisticRegression,0.0045,0.6000,0.7267,0.3841,0.8190,0.9381,0.9603,0.8535,4.9587
5,fusion,raw_input_text,text-embedding-3-large,32,32,0.5909,MLP,0.0045,0.6333,0.7181,0.3071,0.7603,0.8714,0.9111,0.7997,5.8810
1,fusion,raw_input_text,text-embedding-3-large,32,32,0.5909,XGBoost,0.0049,0.5254,0.6804,0.2178,0.7286,0.8492,0.8889,0.7620,8.0651
7,fusion,raw_input_text,text-embedding-3-large,32,32,0.5909,TabTransformer,0.0063,0.5460,0.6778,0.3451,0.8302,0.9206,0.9492,0.8572,5.1984
8,fusion,raw_input_text,text-embedding-3-large,32,32,0.5909,FTTransformer,0.0086,0.3778,0.4897,0.1949,0.4825,0.7127,0.7857,0.5958,13.2492
6,fusion,raw_input_text,text-embedding-3-large,32,32,0.5909,TabNet,0.0164,0.1413,0.3280,0.1598,0.5254,0.5794,0.6206,0.5621,21.2143
4,fusion,raw_input_text,text-embedding-3-large,32,32,0.5909,SVC,0.0596,0.0000,0.1613,0.2479,0.7683,0.8952,0.9190,0.7997,6.7857


## 保存合并结果


In [10]:
# 汇总所有模型结果，保存到单个 use_case1_diagnosis.xlsx 工作簿。
result_frames = []
for frame in [structured_results, embedding_results, fusion_results]:
    if frame is not None and not frame.empty:
        result_frames.append(frame)

if not result_frames:
    raise RuntimeError("No result frames were generated. Enable at least one RUN_*_MODELS flag.")

combined_results = pd.concat(result_frames, ignore_index=True, sort=False)
ordered_cols = [
    "feature_set",
    "text_variant",
    "embedding_model",
    "requested_pca_components",
    "pca_components",
    "pca_explained_variance",
    "model",
] + METRIC_COLS
ordered_cols = [col for col in ordered_cols if col in combined_results.columns]
combined_results = combined_results[ordered_cols]

# 将诊断输出整理为论文表格格式：删除不展示的指标，并把比例指标转为 xx.xx%。
PAPER_METRIC_COLS = ["micro_f1", "macro_f1", "top1", "top3", "top5"]
PAPER_DROP_METRICS = ["hamming_loss", "exact_match", "lrap", "coverage"]

def format_percent_value(value):
    if value is None or pd.isna(value):
        return ""
    if isinstance(value, str):
        text = value.strip()
        if not text or text.lower() in {"nan", "none", "na", "n/a"}:
            return ""
        if text.endswith("%"):
            return text
        value = float(text)
    return f"{float(value):.2%}"

def make_diagnosis_paper_ready(df):
    df = df.copy()
    df = df.drop(columns=[col for col in PAPER_DROP_METRICS if col in df.columns])
    for col in PAPER_METRIC_COLS:
        if col in df.columns:
            df[col] = df[col].map(format_percent_value)
    return df

best_rows = []
for metric in PAPER_METRIC_COLS:
    if metric not in combined_results.columns:
        continue
    values = pd.to_numeric(combined_results[metric], errors="coerce")
    if not values.notna().any():
        continue
    best_idx = values.idxmin() if metric in LOWER_IS_BETTER else values.idxmax()
    best_rows.append(
        {
            "metric": metric,
            "best_direction": "lower" if metric in LOWER_IS_BETTER else "higher",
            **combined_results.loc[best_idx].to_dict(),
        }
    )
best_summary = pd.DataFrame(best_rows)
run_config_df = pd.DataFrame([
    {"parameter": "source_file", "value": str(SOURCE_FILE)},
    {"parameter": "rotd_file", "value": str(ROTD_FILE)},
    {"parameter": "split_file", "value": str(SPLIT_FILE)},
    {"parameter": "result_file", "value": str(DIAGNOSIS_RESULT_FILE)},
    {"parameter": "min_support", "value": MIN_SUPPORT},
    {"parameter": "test_size", "value": TEST_SIZE},
    {"parameter": "random_state", "value": RANDOM_STATE},
    {"parameter": "n_samples", "value": len(data)},
    {"parameter": "n_train", "value": len(train_idx)},
    {"parameter": "n_test", "value": len(test_idx)},
    {"parameter": "n_disease_labels", "value": len(valid_disease_cols)},
    {"parameter": "label_source", "value": "disease_codes 展开后的 ORPHA/ICD-11 标签"},
    {"parameter": "threshold_rule", "value": "每个疾病标签在训练集上单独搜索 F1 最优阈值"},
    {"parameter": "embedding_text_fields", "value": "Past_medical_history, Clinical_reports, Tumor_size"},
    {"parameter": "structured_features", "value": "age, Gender, Ethnicity, Country, Laterality, Phenotype_ids, Anatomical_location_ids, " + ", ".join(GENETIC_COLS)},
    {"parameter": "genetic_feature_columns", "value": ", ".join(GENETIC_COLS)},
    {"parameter": "embedding_models", "value": ", ".join(EMBED_MODELS)},
    {"parameter": "fusion_embedding_models", "value": ", ".join(FUSION_EMBED_MODELS)},
    {"parameter": "pca_components", "value": ", ".join(map(str, PCA_N_COMPONENTS_LIST))},
])

combined_results_paper = make_diagnosis_paper_ready(combined_results)
structured_results_paper = make_diagnosis_paper_ready(structured_results) if not structured_results.empty else structured_results
embedding_results_paper = make_diagnosis_paper_ready(embedding_results) if not embedding_results.empty else embedding_results
fusion_results_paper = make_diagnosis_paper_ready(fusion_results) if not fusion_results.empty else fusion_results
best_summary_paper = make_diagnosis_paper_ready(best_summary)

with pd.ExcelWriter(DIAGNOSIS_RESULT_FILE, engine="openpyxl") as writer:
    combined_results_paper.to_excel(writer, sheet_name="All_results", index=False)
    if not structured_results.empty:
        structured_results_paper.to_excel(writer, sheet_name="Structured_only", index=False)
    if not embedding_results.empty:
        embedding_results_paper.to_excel(writer, sheet_name="Embedding_only", index=False)
    best_summary_paper.to_excel(writer, sheet_name="Best_by_metric", index=False)
    run_config_df.to_excel(writer, sheet_name="Run_config", index=False)

    workbook = writer.book
    bold_font = Font(bold=True)
    for sheet_name in workbook.sheetnames:
        sheet = workbook[sheet_name]
        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions
        for header_cell in sheet[1]:
            header_cell.font = bold_font

    sheet = workbook["All_results"]
    header_to_col = {cell.value: cell.column for cell in sheet[1]}
    model_col = header_to_col.get("model")
    for metric in METRIC_COLS:
        metric_col = header_to_col.get(metric)
        if metric_col is None:
            continue
        values = pd.to_numeric(combined_results[metric], errors="coerce")
        if not values.notna().any():
            continue
        best_value = values.min() if metric in LOWER_IS_BETTER else values.max()
        for idx in values[values.eq(best_value)].index:
            excel_row = idx + 2
            sheet.cell(row=excel_row, column=metric_col).font = bold_font
            if model_col is not None:
                sheet.cell(row=excel_row, column=model_col).font = bold_font

print(f"Saved diagnosis results to {DIAGNOSIS_RESULT_FILE}")
display(best_summary.round(4))


Saved diagnosis results to results\use_case1_diagnosis.xlsx


,metric,best_direction,feature_set,text_variant,embedding_model,requested_pca_components,pca_components,pca_explained_variance,model,hamming_loss,exact_match,micro_f1,macro_f1,top1,top3,top5,lrap,coverage
0,micro_f1,higher,fusion,raw_input_text,text-embedding-3-large,32,32,0.5909,RandomForest,0.0038,0.6254,0.7440,0.3092,0.7873,0.8651,0.8984,0.8057,7.1810
1,macro_f1,higher,fusion,raw_input_text,text-embedding-3-large,32,32,0.5909,LogisticRegression,0.0045,0.6000,0.7267,0.3841,0.8190,0.9381,0.9603,0.8535,4.9587
2,top1,higher,fusion,raw_input_text,text-embedding-3-large,32,32,0.5909,TabTransformer,0.0063,0.5460,0.6778,0.3451,0.8302,0.9206,0.9492,0.8572,5.1984
3,top3,higher,fusion,raw_input_text,text-embedding-3-large,32,32,0.5909,LogisticRegression,0.0045,0.6000,0.7267,0.3841,0.8190,0.9381,0.9603,0.8535,4.9587
4,top5,higher,fusion,raw_input_text,text-embedding-3-large,32,32,0.5909,LogisticRegression,0.0045,0.6000,0.7267,0.3841,0.8190,0.9381,0.9603,0.8535,4.9587
